In [1]:
# Importar las librerías necesarias
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix

In [4]:
# URL del archivo en el bucket público de AWS
url = "https://hybridge-education-machine-learning-datasets.s3.us-east-1.amazonaws.com/Fraud.csv"
df = pd.read_csv(url)

In [5]:
df.head()

,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud
0,1,PAYMENT,9839.64,C1231006815,170136.0,160296.36,M1979787155,0.0,0.0,0,0
1,1,PAYMENT,1864.28,C1666544295,21249.0,19384.72,M2044282225,0.0,0.0,0,0
2,1,TRANSFER,181.00,C1305486145,181.0,0.00,C553264065,0.0,0.0,1,0
3,1,CASH_OUT,181.00,C840083671,181.0,0.00,C38997010,21182.0,0.0,1,0
4,1,PAYMENT,11668.14,C2048537720,41554.0,29885.86,M1230701703,0.0,0.0,0,0


In [7]:
len(df)

6362620

# ¡¡¡6,362,620 OBSERVACIONES!!!

In [6]:
df.shape


(6362620, 11)

In [8]:
df["isFraud"].value_counts()

isFraud
0    6354407
1       8213
Name: count, dtype: int64

In [9]:
df.isnull().sum()

step              0
type              0
amount            0
nameOrig          0
oldbalanceOrg     0
newbalanceOrig    0
nameDest          0
oldbalanceDest    0
newbalanceDest    0
isFraud           0
isFlaggedFraud    0
dtype: int64

In [10]:
df.dtypes

step                int64
type                  str
amount            float64
nameOrig              str
oldbalanceOrg     float64
newbalanceOrig    float64
nameDest              str
oldbalanceDest    float64
newbalanceDest    float64
isFraud             int64
isFlaggedFraud      int64
dtype: object

In [11]:
df["type"].value_counts()


type
CASH_OUT    2237500
PAYMENT     2151495
CASH_IN     1399284
TRANSFER     532909
DEBIT         41432
Name: count, dtype: int64

In [12]:
pd.crosstab(df["type"], df["isFraud"])

isFraud,0,1
type,,
CASH_IN,1399284,0
CASH_OUT,2233384,4116
DEBIT,41432,0
PAYMENT,2151495,0
TRANSFER,528812,4097


In [13]:
df.columns.tolist()


['step',
 'type',
 'amount',
 'nameOrig',
 'oldbalanceOrg',
 'newbalanceOrig',
 'nameDest',
 'oldbalanceDest',
 'newbalanceDest',
 'isFraud',
 'isFlaggedFraud']

In [14]:
X = df.drop(columns=["isFraud", "isFlaggedFraud", "nameOrig", "nameDest"])
y = df["isFraud"]

print("Variables X:", X.shape)
print("Variable y:", y.shape)

Variables X: (6362620, 7)
Variable y: (6362620,)


In [15]:
X = pd.get_dummies(X, columns=["type"], drop_first=True, dtype="int8")

print(X.shape)
print(X.columns.tolist())

(6362620, 10)
['step', 'amount', 'oldbalanceOrg', 'newbalanceOrig', 'oldbalanceDest', 'newbalanceDest', 'type_CASH_OUT', 'type_DEBIT', 'type_PAYMENT', 'type_TRANSFER']


In [16]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Entrenamiento:", X_train.shape)
print("Prueba:", X_test.shape)

Entrenamiento: (5090096, 10)
Prueba: (1272524, 10)


In [17]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Entrenamiento:", X_train_scaled.shape)
print("Prueba:", X_test_scaled.shape)

Entrenamiento: (5090096, 10)
Prueba: (1272524, 10)


In [18]:
from sklearn.linear_model import LogisticRegression

modelo = LogisticRegression(
    max_iter=300,
    class_weight="balanced",
    random_state=42
)

modelo.fit(X_train_scaled, y_train)

print("¡Modelo entrenado correctamente!")

¡Modelo entrenado correctamente!


In [19]:
from sklearn.metrics import classification_report, confusion_matrix

y_pred = modelo.predict(X_test_scaled)

print("REPORTE DE CLASIFICACIÓN")
print(classification_report(y_test, y_pred, zero_division=0))

print("\nMATRIZ DE CONFUSIÓN")
print(confusion_matrix(y_test, y_pred))

REPORTE DE CLASIFICACIÓN
              precision    recall  f1-score   support

           0       1.00      0.95      0.97   1270881
           1       0.02      0.96      0.05      1643

    accuracy                           0.95   1272524
   macro avg       0.51      0.96      0.51   1272524
weighted avg       1.00      0.95      0.97   1272524


MATRIZ DE CONFUSIÓN
[[1206227   64654]
 [     64    1579]]


In [20]:
prob_fraude = modelo.predict_proba(X_test_scaled)[:, 1]

y_pred_90 = (prob_fraude >= 0.90).astype(int)

print("EVALUACIÓN CON UMBRAL DE 90 %")
print(classification_report(y_test, y_pred_90, zero_division=0))

print("\nMATRIZ DE CONFUSIÓN")
print(confusion_matrix(y_test, y_pred_90))

EVALUACIÓN CON UMBRAL DE 90 %
              precision    recall  f1-score   support

           0       1.00      0.99      1.00   1270881
           1       0.12      0.77      0.21      1643

    accuracy                           0.99   1272524
   macro avg       0.56      0.88      0.60   1272524
weighted avg       1.00      0.99      1.00   1272524


MATRIZ DE CONFUSIÓN
[[1261419    9462]
 [    374    1269]]
